In [1]:
import argparse
import json
import re
import os
import torch
import traceback
import numpy as np
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path
from tqdm import tqdm
from argparse import Namespace

# Set correct directory pathing
import os
import sys

# Import project modules
sys.path.insert(0, '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA')
from rdma.rdrag.entity import LLMRDExtractor, RetrievalEnhancedRDExtractor,MultiIterativeRDExtractor,IterativeLLMRDExtractor
from rdma.utils.embedding import EmbeddingsManager
from rdma.hporag.context import ContextExtractor
from rdma.utils.llm_client import LocalLLMClient, APILLMClient
from rdma.utils.setup import setup_device
from dotenv import load_dotenv

load_dotenv()

/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


True

In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [3]:
SAMPLE_SIZE = 2

### GettingStarted

In [4]:
from pyhealth.datasets.mimic4 import MIMIC4NoteDataset

In [5]:
NOTE_ROOT = '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp'

dataset = MIMIC4NoteDataset(root=NOTE_ROOT, tables=["discharge"])
note_df = dataset.global_event_df.collect().to_pandas()

Using default note config: /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/configs/mimic4_note.yaml
Memory usage Before initializing mimic4_note: 564.7 MB
Initializing mimic4_note dataset from /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp (dev mode: False)
Memory usage After initializing mimic4_note: 564.9 MB
No cache_dir provided. Using default cache dir: /Users/williampang/Library/Caches/pyhealth/d41554f3-03fc-5ed2-a6e5-99b796e1ae37


/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/mimic4.py:103: UserWarning: Events from discharge table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


In [6]:
patient_notes = (
    note_df
    .sort_values("timestamp")
    .groupby("patient_id")
    .apply(
        lambda x: dict(zip(x["timestamp"], x["discharge/text"]))
     )
    )

samples = patient_notes.sample(n=SAMPLE_SIZE, random_state=42)

/var/folders/zl/lm3qrxjd2jl17byd443y505h0000gn/T/ipykernel_56098/4148831802.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [7]:
final_notes = {
    str(patient_id): {
        str(charttime): {"note_content": text} 
        for charttime, text in notes.items()
    }
    for patient_id, notes in samples.items()
}

## Extract Rare Disease

In [8]:
args = Namespace(
      llm_type="api",
      api_config="api_config.json"
  )

def initialize_llm_client(args: argparse.Namespace):
    """Initialize appropriate LLM client based on arguments."""
    if args.llm_type == "api":
        if args.api_config:
            return APILLMClient.from_config(args.api_config)
        else:
            return APILLMClient.initialize_from_input()
    else:  # local
        return LocalLLMClient(
            model_type=args.model_type,
            device=device,
            cache_dir=args.cache_dir,
            temperature=args.temperature
        )

llm_client = initialize_llm_client(args)        

entity_extractor = LLMRDExtractor(
      llm_client=llm_client,
      system_message="You are a medical expert specializing in rare diseases."
  )

## Getting context from entities

In [9]:
class CustomContextExtractor:
    def extract_sentences(self, text: str) -> List[str]:
        
        # First split by common sentence terminators while preserving them
        sentence_parts = []
        for part in re.split(r"([.!?])", text):
            if part.strip():
                if part in ".!?":
                    if sentence_parts:
                        sentence_parts[-1] += part
                else:
                    sentence_parts.append(part.strip())

        # Then handle other clinical note delimiters like line breaks and semicolons
        sentences = []
        for part in sentence_parts:
            # Split by semicolons and newlines
            for subpart in re.split(r"[;\n]", part):
                if subpart.strip():
                    sentences.append(subpart.strip())

        return sentences

    def find_entity_context(self, entity: str, sentences: List[str], window_size):
        entity_lower = entity.lower()
        for i, sentence in enumerate(sentences):
            if entity_lower in sentence.lower():
                # Found exact match - include surrounding sentences based on window_size
                return self.get_context_window(sentences, i, window_size)

        # More sophisitication: Use some sort of fuzzy matching. 
        # This requires iterating over each sentence, and then checking
        # if each word in the sentence "fuzzy" matches the entity. If it fuzzy matches above
        # a threshold, then we have found a match and we keep the index (where the sentence came
        # from)

        # #### Psuedocode ####
        # entity_words = set(re.findall(r"\b\w+\b", entity_lower))
        # best_score = 0
        # for i, sentence in enumerate(sentences):
        #     sentence_words = set(re.findall(r"\b\w+\b"), entity_lower)

        #     common_words = entity_words & sentence_words

        #     # Calculate Jaccard similarity as an example
        #     similarity_score = len(common_words)/ (
        #         len(entity_words) + len(sentence_words) - len(common_words)
        #     )

        #     if score > best_score:
        #         best_score = score
        #         best_match_i = i # This updates as a better match gets found

        #  if best_match_index >=0:
        #     return self.get_context_window(sentences, best_match_i, window_size)
        #  return None

    def get_context_window(self, sentences: List[str], center_index: int, window_size: int):
        start_index = max(0, center_index - window_size)
        end_index = min(len(sentences) - 1, center_index + window_size)
        context_sentences = sentences[start_index : end_index + 1]

        return " ".join(context_sentences).strip()

    def extract_context(self, entities: List[str], text: str, window_size: int = 0):

        sentences = self.extract_sentences(text)

        results = []
        for entity in entities:
            context = self.find_entity_context(entity, sentences, window_size)
            results.append(
                {
                    "entity": entity,
                    "context": context or "",  # Empty string if no context found
                }
            )

        return results

In [10]:
context_extractor = CustomContextExtractor()

for i, (patient_id, patient_data) in enumerate(tqdm(list(final_notes.items()), desc="Processing cases")):
    for charttime, note in patient_data.items():
        note['llm_extracted_entities'] = entity_extractor.extract_entities(note['note_content'])
        note['entity_context'] = context_extractor.extract_context(note['llm_extracted_entities'], note['note_content'], window_size=0)

Processing cases:   0%|                                                                                                                | 0/2 [00:00<?, ?it/s]

TOTAL_TOKENS_USED before query: 0


Processing cases:  50%|████████████████████████████████████████████████████                                                    | 1/2 [00:01<00:01,  1.00s/it]

TOTAL_TOKENS_USED before query: 3975


Processing cases: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.13it/s]


In [11]:
# test_case = final_notes['13106750']['2117-04-16 00:00:00']['entity_context']
# print(test_case)

## Verifying whether disease `is_rare_disease`

In [12]:
from fastembed import TextEmbedding
import faiss

class CustomFastembedRDVerifier:
    def __init__(self, model_name: str, llm_client, system_message):
        self.model = TextEmbedding(model_name=model_name)
        self.llm_client = llm_client
        self.system_message = system_message
        sample_embedding = list(self.model.embed(["sample text"]))[0]
        self._dimension = len(sample_embedding)

    def query_text(self, text: str) -> np.ndarray:
        """For FastEmbed, query embedding uses same method as document embedding."""
        return self._get_embedding(text)

    def _get_embedding(self, text: str) -> np.ndarray:
        """Takes one piece of text and returns its vector embedding as a NumPy array of float32s"""
        return np.array(list(self.model.embed([text]))[0]).astype(np.float32)

    def create_index_from_embeddings(self, embeddings_file: np.ndarray):
        self.embedded_documents = np.load(embeddings_file, allow_pickle=True)
        print(f"Loaded {len(self.embedded_documents)} documents")

        embeddings_list = [
            np.array(doc["embedding"])
            for doc in self.embedded_documents
            if isinstance(doc["embedding"], np.ndarray) and doc["embedding"].size > 0
        ]
        
        embeddings_array = np.vstack(embeddings_list).astype(np.float32)

        #Create a FAISS index for the embeddings
        dimension = embeddings_array.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings_array)

    def search(
        self, query, k
    ):
        """
        Search for similar documents using a query.
        """
        if isinstance(query, str):
            query_vector = self.query_text(query).reshape(1, -1)
        else:
            query_vector = query.reshape(1, -1)

        distances, indices = self.index.search(query_vector, k)

        return distances, indices

    def verify_rare_disease(self, term, embeddings_file):
        self.create_index_from_embeddings(embeddings_file)
        distances, indices = self.search(term, k=5)
        context = "\nPotential matches from database:\n" + "\n".join(
            f"{i+1}. {self.embedded_documents[idx].get('name')} (dist: {dist:.4f})"
            for i, (idx, dist) in enumerate(zip(indices[0], distances[0]))
        )

        prompt = f"""Analyze this medical term and determine if it represents a rare disease.

        Term: {term}
        {context}

        A term should ONLY be considered a rare disease if ALL these criteria are met:
        1. It is a disease or syndrome (not just a symptom, finding, or condition)
        2. It is rare (affecting less than 1 in 2000 people)
        3. There is clear evidence in the context or term itself indicating rarity
        4. For variants of common diseases, it must be explicitly marked as a rare variant
        5. The term should align with the type of entries in our rare disease database.
        6. If there is a partial match, i.e cholangitis vs. sclerosing cholangitis. There must be a mention of its descriptor (sclerosing) in the term itself, otherwise it's invalid match.

        Response format:
        First line: "DECISION: true" or "DECISION: false"
        Next lines: Brief explanation of decision"""
         
        response = self.llm_client.query(prompt, self.system_message).strip().lower()

        # print("Prompt:")
        # print(prompt)
        # print("Response:")
        # print(response)
        # print("-----------------\n")

        return "decision: true" in response.lower()

    @property
    def dimension(self) -> int:
        return self._dimension

In [13]:
verifier = CustomFastembedRDVerifier(model_name="BAAI/bge-small-en-v1.5", 
                                 llm_client=llm_client,
                                 system_message="You are a medical expert specializing in rare diseases.")
embeddings_file = "/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp/EmbeddedDocs/rd_orpha_medembed.npy"
# verifier.create_index_from_embeddings(embeddings_file)
verifier.verify_rare_disease("Fabry disease", embeddings_file)

verified_notes = {}

for patient_id, patient_data in tqdm(final_notes.items(), desc="Processing patients", total=len(final_notes)):
    verified_notes[str(patient_id)] = {}

    for charttime, note in patient_data.items():
        entities = note["llm_extracted_entities"]
        contexts = note["entity_context"]

        enriched = []
        for j, ent in enumerate(entities):
            is_rare = verifier.verify_rare_disease(ent, embeddings_file)
            ent_context = contexts.get(ent) if isinstance(contexts, dict) else contexts[j]

            enriched.append({
                "is_rare_disease": bool(is_rare),
                "entity_context": ent_context,
            })

        verified_notes[str(patient_id)][str(charttime)] = {
            "note_content": note.get("note_content"),
            "entities": enriched,
        }


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded 26750 documents
TOTAL_TOKENS_USED before query: 8168


Processing patients:   0%|                                                                                                             | 0/2 [00:00<?, ?it/s]

Loaded 26750 documents
TOTAL_TOKENS_USED before query: 8675
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 9171
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 9687
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 10167
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 10643
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 11120
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 11589
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 12093
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 12566
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 13105
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 13608
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 14103
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 14576
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 15107
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 15625
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 16116
Loaded 26750 documents
TOTA

Processing patients:  50%|██████████████████████████████████████████████████▌                                                  | 1/2 [00:15<00:15, 15.06s/it]

Loaded 26750 documents
TOTAL_TOKENS_USED before query: 18304
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 18802
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 19261
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 19758
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 20257
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 20782
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 21283
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 21748
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 22281
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 22776
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 23272
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 23752
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 24276
Loaded 26750 documents
TOTAL_TOKENS_USED before query: 24819


Processing patients: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:51<00:00, 25.84s/it]


In [14]:
verified_notes['13106750']['2117-04-16 00:00:00']['entities']

[{'is_rare_disease': False,
  'entity_context': {'entity': 'Squamos cell carcinoma',
   'context': 'Squamos cell carcinoma of epiglottis, treated with'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'Adenocarcinoma',
   'context': 'completed ___ ___, adenocarcinoma of stage IV left lung'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'COPD', 'context': 'COPD'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'CKD',
   'context': 'CKD stage III (baseline 1.'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'HIT',
   'context': 'HIT with positive antibody assay.'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'Hypertension', 'context': 'Hypertension'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'Hyperlipidemia', 'context': 'Hyperlipidemia'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'GERD', 'context': 'GERD'}},
 {'is_rare_disease': False,
  'entity_context': {'entity': 'Uric acid neph